In [4]:
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA


# 1. Load Data from NPZ
def load_data(file_path):
    data = np.load(file_path)

    print("Available keys:", data.files)

    X = data['features']
    y = data['labels']

    print("X Shape:", X.shape)
    print("y Shape:", y.shape)

    return X, y


# 2. Preprocess Data
def preprocess_data(X, y):
    # Encode labels if needed
    if y.dtype == object:
        y = LabelEncoder().fit_transform(y)

    return X, y


# 3. Split Data
def split_data(X, y):
    return train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


# 4. Apply PCA
def apply_pca(X_train, X_test, n_components=200):
    pca = PCA(n_components=n_components, random_state=42)

    X_train_pca = pca.fit_transform(X_train)
    X_test_pca = pca.transform(X_test)

    print("Explained variance ratio sum:", sum(pca.explained_variance_ratio_))

    return X_train_pca, X_test_pca


# 5. Hyperparameter Tuning
def tune_model(X_train, y_train):

    rf = RandomForestClassifier(
        random_state=42,
        class_weight="balanced"
    )

    param_dist = {
        'n_estimators': [200, 300, 500],
        'max_depth': [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt']
    }

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    random_search = RandomizedSearchCV(
        estimator=rf,
        param_distributions=param_dist,
        n_iter=10,
        cv=cv,
        scoring='accuracy',
        n_jobs=-1,
        random_state=42,
        verbose=1
    )

    random_search.fit(X_train, y_train)

    print("\nBest Parameters:", random_search.best_params_)
    print("Best CV Score:", random_search.best_score_)

    return random_search.best_estimator_


# 6. Evaluation
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)

    print("\nTest Accuracy:", accuracy_score(y_test, y_pred))
    print("\nClassification Report:\n",
          classification_report(y_test, y_pred, zero_division=0))


# 7. Run Pipeline
def run_pipeline(file_path):
    X, y = load_data(file_path)

    X, y = preprocess_data(X, y)

    X_train, X_test, y_train, y_test = split_data(X, y)

    # Apply PCA
    X_train, X_test = apply_pca(X_train, X_test, n_components=200)

    # Train model
    model = tune_model(X_train, y_train)

    # Evaluate
    evaluate_model(model, X_test, y_test)


# 🚀 Execute
run_pipeline("fc7_features.npz")

Available keys: ['features', 'labels', 'label_names']
X Shape: (633, 4096)
y Shape: (633,)
Explained variance ratio sum: 0.8808567
Fitting 3 folds for each of 10 candidates, totalling 30 fits

Best Parameters: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 20}
Best CV Score: 0.3557222691838076

Test Accuracy: 0.3700787401574803

Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.50      0.25      0.33         4
           2       0.00      0.00      0.00        14
           3       1.00      0.17      0.29         6
           4       0.00      0.00      0.00         6
           5       0.00      0.00      0.00         4
           6       0.80      0.20      0.32        20
           7       1.00      0.08      0.15        12
           8       0.34      0.90      0.50        42
           9       0.40      0.12      0.18 

In [11]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# Optional models
try:
    from xgboost import XGBClassifier
    xgb_available = True
except:
    xgb_available = False

try:
    from catboost import CatBoostClassifier
    catboost_available = True
except:
    catboost_available = False


# 1. Load Data from NPZ
def load_data(file_path):
    data = np.load(file_path)

    print("Available keys:", data.files)

    X = data['features']
    y = data['labels']

    print("X Shape:", X.shape)
    print("y Shape:", y.shape)

    return X, y


# 2. Preprocess Data
def preprocess_data(X, y):
    if y.dtype == object:
        y = LabelEncoder().fit_transform(y)

    return X, y


# 3. Split Data
def split_data(X, y):
    return train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )


# 4. Train & Evaluate Models
def train_and_evaluate(X_train, X_test, y_train, y_test):

    models = {
        "SVM": SVC(),
        "Decision Tree": DecisionTreeClassifier(),
        "Random Forest": RandomForestClassifier(n_estimators=100),
        "AdaBoost": AdaBoostClassifier(n_estimators=100),
        "Naive Bayes": GaussianNB(),
        "MLP": MLPClassifier(max_iter=300)
    }

    # Optional models
    if xgb_available:
        models["XGBoost"] = XGBClassifier(
            eval_metric='logloss',
            n_estimators=100,
            verbosity=0
        )

    if catboost_available:
        models["CatBoost"] = CatBoostClassifier(
            verbose=0,
            iterations=100
        )

    results = []

    for name, model in models.items():
        print(f"Training {name}...")

        model.fit(X_train, y_train)

        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)

        results.append({
            "Model": name,
            "Train Accuracy": accuracy_score(y_train, y_train_pred),
            "Test Accuracy": accuracy_score(y_test, y_test_pred),
            "Precision": precision_score(y_test, y_test_pred, average='weighted', zero_division=0),
            "Recall": recall_score(y_test, y_test_pred, average='weighted', zero_division=0),
            "F1 Score": f1_score(y_test, y_test_pred, average='weighted', zero_division=0)
        })

    return pd.DataFrame(results)


# 5. Run Pipeline
def run_full_model_pipeline(file_path):
    X, y = load_data(file_path)

    X, y = preprocess_data(X, y)

    X_train, X_test, y_train, y_test = split_data(X, y)

    results_df = train_and_evaluate(X_train, X_test, y_train, y_test)

    print("\nFinal Comparison Table:\n")
    print(results_df)
    results_df.to_excel("model_results.xlsx", index=False)

    return results_df

if __name__ == "__main__":
    run_full_model_pipeline("fc7_features.npz")

Available keys: ['features', 'labels', 'label_names']
X Shape: (633, 4096)
y Shape: (633,)
Training SVM...
Training Decision Tree...
Training Random Forest...
Training AdaBoost...
Training Naive Bayes...
Training MLP...
Training XGBoost...
Training CatBoost...

Final Comparison Table:

           Model  Train Accuracy  Test Accuracy  Precision    Recall  F1 Score
0            SVM        0.677866       0.370079   0.304502  0.370079  0.280514
1  Decision Tree        0.992095       0.314961   0.301595  0.314961  0.302383
2  Random Forest        0.992095       0.354331   0.322229  0.354331  0.285879
3       AdaBoost        0.357708       0.299213   0.227770  0.299213  0.249755
4    Naive Bayes        0.909091       0.338583   0.278740  0.338583  0.270479
5            MLP        0.992095       0.393701   0.371300  0.393701  0.365430
6        XGBoost        0.992095       0.385827   0.366225  0.385827  0.358045
7       CatBoost        0.992095       0.362205   0.301312  0.362205  0.319947


PermissionError: [Errno 13] Permission denied: 'model_results.xlsx'